# Gộp các pack vector thành FAISS unified

Notebook chỉ clone dự án và gọi mã trong `src/`. Chọn **Runtime > Change runtime type > GPU** khi chạy mô hình. Chạy các ô từ trên xuống.

**Trước khi chạy:** mã nguồn mới phải có trên GitHub; đặt `GIT_REF` đúng nhánh/tag/commit. Notebook không lấy được các thay đổi chỉ nằm trên máy cá nhân. Không đặt token GitHub trong URL hoặc lưu trong notebook. Dự án riêng tư cần cấu hình xác thực Git của phiên Colab trước.

## 1. Clone dự án
Nếu đổi phiên bản mã sau khi đã import module, khởi động lại phiên Python trước khi tiếp tục.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = "https://github.com/qtamtensor05/ViGovBot.git"
GIT_REF = "codex/feat-multi-embedding-colab"  # Nhánh hiện tại; đổi main sau khi merge hoặc dùng commit cố định.
REPO_DIR = Path("/content/ViGovBot")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("REPO_DIR đang trỏ tới dự án khác; chọn thư mục mới")
    changes = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
    if changes.strip():
        raise RuntimeError("Có thay đổi trong checkout Colab; lưu lại hoặc chọn REPO_DIR mới trước khi cập nhật")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Commit đang chạy:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. Cài thư viện
Cài từ file requirements của đúng phiên bản dự án vừa clone.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-vector-db.txt"], check=True)

## 3. Kết nối Google Drive
Chọn tài khoản chứa dữ liệu và cấp quyền khi Colab yêu cầu.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Kiểm tra danh sách pack và gộp
Không cần GPU, nhưng cần đủ RAM cho ma trận và chỉ mục cuối. Chờ mọi worker hoàn tất trước khi gộp. Với tên `vectors_pack_pack1.npy`, mã tương ứng là `pack_pack1`; sửa danh sách dưới đây theo file thực tế.

In [ ]:
from src.vectordb.merge import discover_pairs
INPUT_DIR = Path("/content/drive/MyDrive/RAG_Data/completed")
OUTPUT_DIR = Path("/content/drive/MyDrive/RAG_Data/unified")
EXPECTED_PACK_IDS = ["pack_pack1", "pack_pack2", "pack_pack3", "pack_pack4", "pack_pack5"]
pairs = discover_pairs(INPUT_DIR)
found = {p.stem.removeprefix("vectors_") for p, _ in pairs}
if found != set(EXPECTED_PACK_IDS):
    raise ValueError(f"Danh sách pack không khớp: đã có {sorted(found)}, cần {EXPECTED_PACK_IDS}")
subprocess.run([sys.executable, "-m", "src.vectordb.merge", str(INPUT_DIR), "--output-dir", str(OUTPUT_DIR)],
               cwd=REPO_DIR, check=True)

Kết quả: `tthc_unified.index` và `tthc_unified_metadata.json`. Notebook RAG nhận trực tiếp thư mục `unified` hoặc ZIP chứa hai file này.